# RusaModels — Qwen3-4B QLoRA training on Google Colab T4

**Purpose:** run the repository's actual SFT training script on a Colab NVIDIA T4 (16 GB), evaluate the resulting adapter, and copy the adapter weights to Google Drive.

## Before running
1. In Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Run the cells from top to bottom without skipping the GPU check.
3. Mount Google Drive when prompted. The finished adapter and evaluation report are copied to `MyDrive/RusaModels/training-runs/qwen3-4b-qlora-t4/`.

**Important:** the repository currently contains a seed dataset (120 synthetic training examples and 36 held-out evaluation examples). This is an initial engineering experiment, not production-scale training data. QLoRA produces adapter weights for the pinned base model; it does not create a new full 4B base checkpoint. A failed evaluation gate does not erase the adapter, but the adapter must remain unpromoted.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. Set Colab Runtime type to T4 GPU, then rerun this cell.')
gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gib:.2f} GiB')
print(f'PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}')
if 't4' not in gpu_name.lower():
    raise RuntimeError(f'Expected NVIDIA T4, but Colab assigned: {gpu_name}')
if vram_gib < 14:
    raise RuntimeError(f'Expected at least 14 GiB visible VRAM, got {vram_gib:.2f} GiB')
print('PASS: NVIDIA T4 runtime confirmed.')


In [ ]:
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')

RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
PROJECT_DIR = Path('/content/RusaModels')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/RusaModels/training-runs') / f'qwen3-4b-qlora-t4-{RUN_ID}'
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / '.git').exists():
        raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git repository; remove or rename it, then rerun.')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/StrategyG/RusaModels.git', str(PROJECT_DIR)], check=True)

commit = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Run ID:', RUN_ID)
print('Repository:', PROJECT_DIR)
print('Commit:', commit)
print('Drive output:', DRIVE_OUTPUT_DIR)
assert (PROJECT_DIR / 'configs/training/agent-sft-t4.yaml').is_file()
assert (PROJECT_DIR / 'rusamodels/training/train_sft.py').is_file()


In [ ]:
print('Installing the repository training dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_DIR / 'requirements-training.txt')], check=True)
print('Dependency installation finished.')


In [ ]:
TRAIN_FILE = PROJECT_DIR / 'data/agent_sft/train.jsonl'
EVAL_FILE = PROJECT_DIR / 'data/agent_sft/eval.jsonl'
CONFIG_FILE = PROJECT_DIR / 'configs/training/agent-sft-t4.yaml'
LOCAL_OUTPUT_DIR = PROJECT_DIR / 'artifacts' / f'rusa-agent-qwen3-4b-qlora-t4-{RUN_ID}'
ADAPTER_DIR = LOCAL_OUTPUT_DIR / 'adapter'
EVAL_REPORT = LOCAL_OUTPUT_DIR / 'agent-sft-evaluation-t4.json'

subprocess.run([
    sys.executable, str(PROJECT_DIR / 'rusamodels/training/validate_dataset.py'),
    '--train-file', str(TRAIN_FILE), '--eval-file', str(EVAL_FILE)
], cwd=PROJECT_DIR, check=True)
print('PASS: curated train/eval split validated.')


## Train the adapter

The cell below invokes `rusamodels/training/train_sft.py` from the repository. It uses the pinned Qwen3 model revision and the T4 config: NF4 4-bit quantization, FP16 compute, LoRA rank 8, batch size 1, gradient accumulation 16, 3 epochs, and a 2,048-token maximum sequence length. Training checkpoints and the final adapter are first written to the Colab VM; the next cells copy the finished artifacts to Drive.

In [ ]:
train_cmd = [
    sys.executable, '-u', str(PROJECT_DIR / 'rusamodels/training/train_sft.py'),
    '--config', str(CONFIG_FILE),
    '--output-dir', str(LOCAL_OUTPUT_DIR)
]
print('Starting actual QLoRA training...')
print('Command:', ' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_DIR, check=True)

required = [ADAPTER_DIR / 'adapter_config.json', ADAPTER_DIR / 'adapter_model.safetensors', ADAPTER_DIR / 'rusa-finetune-provenance.json']
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Training did not produce all required adapter files: ' + ', '.join(missing))
print('PASS: adapter weights and provenance were written.')


## Evaluate on the held-out split

The evaluator writes its JSON report before enforcing the configured quality gates. If a gate fails, the notebook keeps the report and continues to package the candidate weights, but it marks the result as **not passed**. Do not deploy an adapter unless the gates pass and it has completed further review.

In [ ]:
eval_cmd = [
    sys.executable, '-u', str(PROJECT_DIR / 'rusamodels/evaluation/evaluate_agent_sft.py'),
    '--adapter-dir', str(ADAPTER_DIR),
    '--eval-file', str(EVAL_FILE),
    '--config', str(CONFIG_FILE),
    '--output', str(EVAL_REPORT)
]
print('Running held-out evaluation...')
eval_result = subprocess.run(eval_cmd, cwd=PROJECT_DIR)
EVALUATION_PASSED = (eval_result.returncode == 0 and EVAL_REPORT.is_file())
if EVAL_REPORT.is_file():
    report = json.loads(EVAL_REPORT.read_text())
    print(json.dumps(report.get('metrics', {}), indent=2))
else:
    report = None
    print('WARNING: evaluation did not produce a report.')
print('Evaluation gate:', 'PASS' if EVALUATION_PASSED else 'NOT PASSED')


In [ ]:
if not ADAPTER_DIR.is_dir():
    raise FileNotFoundError(f'Adapter directory not found: {ADAPTER_DIR}')
DRIVE_ADAPTER_DIR = DRIVE_OUTPUT_DIR / 'adapter'
shutil.copytree(ADAPTER_DIR, DRIVE_ADAPTER_DIR)

for source_name, destination_name in [
    ('training_config.yaml', 'training_config.yaml'),
    ('checkpoints/train_results.json', 'train_results.json'),
    ('checkpoints/eval_results.json', 'trainer_eval_results.json'),
]:
    source = LOCAL_OUTPUT_DIR / source_name
    if source.is_file():
        shutil.copy2(source, DRIVE_OUTPUT_DIR / destination_name)
if EVAL_REPORT.is_file():
    shutil.copy2(EVAL_REPORT, DRIVE_OUTPUT_DIR / EVAL_REPORT.name)

archive_base = DRIVE_OUTPUT_DIR / 'rusa-agent-qwen3-4b-qlora-t4-adapter'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=DRIVE_ADAPTER_DIR)
summary = {
    'run_id': RUN_ID,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'repository_commit': commit,
    'gpu': gpu_name,
    'vram_gib': round(vram_gib, 2),
    'evaluation_passed': EVALUATION_PASSED,
    'adapter_directory': str(DRIVE_ADAPTER_DIR),
    'adapter_zip': archive_path,
    'adapter_zip_bytes': Path(archive_path).stat().st_size,
}
(DRIVE_OUTPUT_DIR / 'colab-run-summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
print('\nAdapter files:')
for path in sorted(DRIVE_ADAPTER_DIR.iterdir()):
    if path.is_file():
        print(f'{path.name}: {path.stat().st_size:,} bytes')


## If the T4 runs out of memory

Do not silently truncate training examples. The training script intentionally fails if a prompt plus completion exceeds the configured sequence length. If the first run fails specifically with CUDA out-of-memory, start a separate experiment with a reviewed fallback config: set `dataset.max_seq_length` to `1024`, `lora.r` to `4`, and `lora.alpha` to `8`, and write to a different output directory such as `rusa-agent-qwen3-4b-qlora-t4-fallback`. Re-run dataset validation and inspect sequence lengths before training. Keep the original run and its logs for traceability.

## Result checklist

- [ ] Colab reports NVIDIA T4 and at least 14 GiB visible VRAM.
- [ ] Dataset validation passes.
- [ ] Training completes and `adapter/adapter_model.safetensors` exists.
- [ ] `adapter/rusa-finetune-provenance.json` records GPU, model revision, dataset hashes, and training metrics.
- [ ] Held-out evaluation report is saved; review whether all gates passed.
- [ ] Adapter folder and ZIP are present in Google Drive.

The adapter is a candidate artifact. The current dataset is synthetic and small; it is not sufficient by itself to establish production readiness.